### About the dataset

This Online Retail II data set contains all the transactions occurring for a UK-based and registered, non-store online retail between 01/12/2009 and 09/12/2011.The company mainly sells unique all-occasion gift-ware. Many customers of the company are wholesalers

I'll use unsupervised learning algorithms to find the hidden patterns of dataset, to use them later in business for the customer type clustering.

E.g. company can use it to understand which customers are at risk of quiting the service and it'll help company to start reactivation campaign or stop spending budget on them.

On the other side if customer is buying a lot, company can make personalized offer to increase their products selling even more.

---

### Import & Basic analysis

In [101]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [102]:
df = pd.read_csv("../data/raw/online_retail_II.csv")

In [103]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [104]:
df.describe(include="all")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371,1067371,1062989,1.067371e+06,1067371,1.067371e+06,824364.000000,1067371
unique,53628,5305,5698,NaN,47635,NaN,NaN,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,2010-12-06 16:57:00,NaN,NaN,United Kingdom
freq,1350,5829,5918,NaN,1350,NaN,NaN,981330
mean,NaN,NaN,NaN,9.938898e+00,NaN,4.649388e+00,15324.638504,NaN
std,NaN,NaN,NaN,1.727058e+02,NaN,1.235531e+02,1697.464450,NaN
min,NaN,NaN,NaN,-8.099500e+04,NaN,-5.359436e+04,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000e+00,NaN,1.250000e+00,13975.000000,NaN
50%,NaN,NaN,NaN,3.000000e+00,NaN,2.100000e+00,15255.000000,NaN
75%,NaN,NaN,NaN,1.000000e+01,NaN,4.150000e+00,16797.000000,NaN


In [105]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [106]:
df.duplicated().sum()

np.int64(34335)

In [107]:
df.shape

(1067371, 8)

### Handling missing values

In [108]:
df = df.drop(columns=['Description'])

`Description feature` doesn't represent any useful information, so i'll drop it

In [109]:
print(f"NA values are {df["Customer ID"].isna().mean().round(3) * 100}% of whole dataset")

NA values are 22.8% of whole dataset


NA values of `Customer ID` will be dropped, even though it's almost 1/4 of dataset.

The reason for it - is that for our task it's must-have to know which customer bought which product, because in other way we won't be able to determine clusters precisely.

In [110]:
df = df.dropna(axis=0)

In [111]:
df.isna().sum().any()

np.False_

### Handling duplicates

In [112]:
df[df.duplicated(keep=False)].sort_values(
    ["Invoice", "StockCode", "Customer ID"]
).head(6)

,Invoice,StockCode,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom


They are represent same information so it's reasonable to drop them to no tdidstort the model.

In [113]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"Removed: {before - after:,}")
print(f"Remaining: {after:,}")

Removed: 26,481
Remaining: 797,883


In [114]:
df.duplicated().any()

np.False_

---

### Handling cancellations and values under zero

If `Invoice` feature's value starts with `c` it means there was a cancellation

In [115]:
df['is_cancelled'] = df['Invoice'].astype(str).str.startswith('C')

In [116]:
df.groupby("is_cancelled")["Quantity"].agg(["count", "mean", "sum"])

,count,mean,sum
is_cancelled,,,
False,779493,13.506731,10528402
True,18390,-25.719195,-472976


I will drop cancelled rows since i want clean dataset without cancellation

In [117]:
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df.drop(columns=['is_cancelled'])

In [118]:
(df["Quantity"] < 0).sum()

np.int64(0)

In [119]:
(df["Price"] <= 0).sum()

np.int64(70)

In [120]:
df[df["Price"] <= 0][
    ["Invoice", "StockCode", "Quantity", "Price"]
].head(12)

,Invoice,StockCode,Quantity,Price
4674,489825,22076,12,0.0
6781,489998,48185,2,0.0
16107,490727,M,1,0.0
18738,490961,22065,1,0.0
18739,490961,22142,12,0.0
32916,492079,85042,8,0.0
40101,492760,21143,12,0.0
47126,493761,79320,24,0.0
48342,493899,22355,10,0.0
57619,494607,21533,12,0.0


Those rows are invalid and must be dropped

In [121]:
df = df[~(df['Price'] <= 0)]

---

### Outliers handling

In [129]:
df["Quantity"].describe()

count    779423.000000
mean         13.489016
std         145.855640
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

In [130]:
df["Price"].describe()

count    779423.000000
mean          3.218488
std          29.676178
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max       10953.500000
Name: Price, dtype: float64

In [132]:
df[["Quantity", "Price"]].quantile(
    [0.01, 0.25, 0.5, 0.75, 0.99]
)

,Quantity,Price
0.01,1.0,0.29
0.25,2.0,1.25
0.50,6.0,1.95
0.75,12.0,3.75
0.99,144.0,14.95
